In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Paths -- edit these for your environment.
RAW_DIR  = "/PATH/TO/raw"              # raw MGB extracts (edw/, clarity/, Datamart/)
DATA_DIR = "/PATH/TO/data"             # project data root; lookup tables live here
WORK_DIR = f"{DATA_DIR}/20250604"      # identified intermediate outputs -- keep on the secure server
DEID_DIR = f"{WORK_DIR}/deidentified"  # de-identified outputs and ID / date-shift maps

In [ ]:
base_cohort = pd.read_csv(f'{WORK_DIR}/microbiology_cultures_cohort.csv')
mrn_map = pd.read_csv(f'{RAW_DIR}/edw/MRN_map/EDW_MRN_map_20240713.csv')
micro = pd.read_csv(f'{RAW_DIR}/Datamart/AllMGB_micro_20150525-20250201.csv')

In [ ]:
# List of target organisms
target_organisms = [
    # Escherichia
    'ESCHERICHIA COLI', 'ESCHERICHIA COLI (CARBAPENEM RESISTANT)', 'ESCHERICHIA FERGUSONII',
    'ESCHERICHIA HERMANII', 'ESCHERICHIA SPECIES', 'ESCHERICHIA VULNERIS',
    # Klebsiella
    'KLEBSIELLA AEROGENES', 'KLEBSIELLA AEROGENES (CARBAPENEM RESISTANT)', 'KLEBSIELLA OXYTOCA',
    'KLEBSIELLA OZAENAE', 'KLEBSIELLA PNEUMONIAE', 'KLEBSIELLA PNEUMONIAE (CARBAPENEM RESISTANT)',
    'KLEBSIELLA PNEUMONIAE SSP. OZAENAE', 'KLEBSIELLA SPECIES', 'KLEBSIELLA OXYTOCA (CARBAPENEM RESISTANT)',
    # Enterococcus
    'ENTEROCOCCUS AVIUM', 'ENTEROCOCCUS CASSELIFLAVUS', 'ENTEROCOCCUS DURANS',
    'ENTEROCOCCUS DURANS/HIRAE', 'ENTEROCOCCUS FAECALIS', 'ENTEROCOCCUS FAECALIS - VANCO RESISTANT',
    'ENTEROCOCCUS FAECIUM', 'ENTEROCOCCUS FAECIUM - VANCO RESISTANT', 'ENTEROCOCCUS FAECIUM -  VANCO RESISTANT',
    'ENTEROCOCCUS GALLINARUM', 'ENTEROCOCCUS RAFFINOSUS', 'ENTEROCOCCUS SPECIES',
    # Coagulase negative Staph
    'COAG NEGATIVE STAPHYLOCOCCUS', 'STAPHYLOCOCCUS SPECIES - COAG NEGATIVE, NOT STAPH SAPROPHYTICUS',
    # Staphylococcus
    'STAPHYLOCOCCUS AUREUS', 'STAPH AUREUS {MRSA}', 'STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE)',
    'STAPHYLOCOCCUS COHNII', 'STAPHYLOCOCCUS EPIDERMIDIS', 'STAPHYLOCOCCUS HAEMOLYTICUS',
    'STAPHYLOCOCCUS LUGDUNENSIS', 'STAPHYLOCOCCUS SAPROPHYTICUS', 'STAPHYLOCOCCUS SPECIES',
    'STAPHYLOCOCCUS SPECIES -NOT STAPH. AUREUS', 'STAPHYLOCOCCUS SPECIES, COAG PENDING',
    # Proteus
    'PROTEUS MIRABILIS', 'PROTEUS SPECIES', 'PROTEUS VULGARIS GROUP',
    'ZZZPROTEUS PENNERI', 'ZZZPROTEUS VULGARIS',
    # Enterobacter
    'ENTEROBACTER AMNIGENUS 1', 'ENTEROBACTER ASBURIAE', 'ENTEROBACTER CANCEROGENUS',
    'ENTEROBACTER CLOACAE', 'ENTEROBACTER CLOACAE COMPLEX', 'ENTEROBACTER CLOACAE COMPLEX (CARBAPENEM RESISTANT)',
    'ENTEROBACTER GERGOVIAE', 'ENTEROBACTER SPECIES', 'ENTEROBACTER TAYLORAE', 'ZZZENTEROBACTER AEROGENES',
    # Citrobacter
    'CITROBACTER AMALONATICUS COMPLEX', 'CITROBACTER BRAAKII', 'CITROBACTER FREUNDII',
    'CITROBACTER FREUNDII COMPLEX', 'CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT)',
    'CITROBACTER KOSERI', 'CITROBACTER KOSERI (CARBAPENEM RESISTANT)', 'CITROBACTER SPECIES',
    'CITROBACTER YOUNGAE', 'ZZZCITROBACTER KOSERI',
    # Streptococcus 
    'STREPTOCOCCUS AGALACTIAE', 'STREPTOCOCCUS AGALACTIAE (GROUP B)', 'STREPTOCOCCUS AGALACTIAE {GROUP B}',
    'STREPTOCOCCUS ANGINOSUS GROUP', 'STREPTOCOCCUS BOVIS/EQUINUS COMPLEX', 'STREPTOCOCCUS CANIS',
    'STREPTOCOCCUS CONSTELLATUS', 'STREPTOCOCCUS DYSGALACTIAE', 'STREPTOCOCCUS DYSGALACTIAE/EQUISIMILIS',
    'STREPTOCOCCUS EQUI', 'STREPTOCOCCUS GORDONII', 'STREPTOCOCCUS GRAM (+) GROUP', 'STREPTOCOCCUS GROUP A',
    'STREPTOCOCCUS GROUP B', 'STREPTOCOCCUS GROUP C', 'STREPTOCOCCUS GROUP D', 'STREPTOCOCCUS GROUP F',
    'STREPTOCOCCUS GROUP G', 'STREPTOCOCCUS INFANTARIUS', 'STREPTOCOCCUS INFANTIS', 'STREPTOCOCCUS INTERMEDIUS',
    'STREPTOCOCCUS MITIS', 'STREPTOCOCCUS MITIS GROUP', 'STREPTOCOCCUS MITIS/STREPTOCOCCUS ORALIS',
    'STREPTOCOCCUS MUTANS', 'STREPTOCOCCUS MUTANS GROUP', 'STREPTOCOCCUS ORALIS', 'STREPTOCOCCUS PARASANGUINIS',
    'STREPTOCOCCUS PNEUMONIAE', 'STREPTOCOCCUS PYOGENES (GROUP A)', 'STREPTOCOCCUS PYOGENES {GROUP A}',
    'STREPTOCOCCUS SALIVARIUS', 'STREPTOCOCCUS SALIVARIUS GROUP', 'STREPTOCOCCUS SANGUINIS',
    'STREPTOCOCCUS SANGUIS', 'STREPTOCOCCUS SPECIES', 'STREPTOCOCCUS SPECIES - ALPHA HEMOLYTIC',
    'STREPTOCOCCUS SPECIES - ALPHA HEMOLYTIC, NOT S.PNEUMO', 'ZZZSTREPTOCOCCUS GALLOLYTICUS SUBSP. GALLOLYTICUS',
    'ZZZSTREPTOCOCCUS GALLOLYTICUS SUBSP. PASTEURIANUS', 'VIRIDANS GROUP STREPTOCOCCI',
    'STREPTOCOCCUS ANGINOSUS', 'STREPTOCOCCUS BOVIS GP.', 'STREPTOCOCCUS GALLOLYTICUS',
    # Pseudomonas
    'MUCOID PSEUDOMONAS AERUGINOSA', 'PSEUDOMONAS AERUGINOSA', 'PSEUDOMONAS AERUGINOSA (NON-MUCOID CF)',
    'PSEUDOMONAS FLUORESCENS', 'PSEUDOMONAS FLUORESCENS GROUP', 'PSEUDOMONAS MENDOCINA',
    'PSEUDOMONAS ORYZIHABITANS', 'PSEUDOMONAS PUTIDA', 'PSEUDOMONAS PUTIDA GROUP',
    'PSEUDOMONAS SPECIES', 'PSEUDOMONAS SPECIES - FLUORESCENS/PUTIDA GROUP',
    'PSEUDOMONAS SPECIES - NOT P.AERUGINOSA', 'PSEUDOMONAS STUTZERI', 'PSEUDOMONAS STUTZERI GROUP',
    # Acinetobacter
    'ACINETOBACTER BAUMANNII', 'ACINETOBACTER BAUMANNII COMPLEX', 'ACINETOBACTER HAEMOLYTICUS',
    'ACINETOBACTER LWOFFI', 'ACINETOBACTER SPECIES',
    # Providencia
    'PROVIDENCIA RETTGERI', 'PROVIDENCIA SPECIES', 'PROVIDENCIA STUARTII',
    # Serratia
    'SERRATIA LIQUEFACIENS', 'SERRATIA MARCESCENS', 'SERRATIA MARCESCENS (CARBAPENEM RESISTANT)',
    'SERRATIA ODORIFERA', 'SERRATIA PLYMUTHICA', 'SERRATIA RUBIDAEA', 'SERRATIA SPECIES',
    # Morganella
    'MORGANELLA MORGANII',
    # Stenotrophomonas
    'STENOTROPHOMONAS MALTOPHILIA'
    # Candida
    'CANDIDA ALBICANS', 'CANDIDA DUBLINIENSIS', 'CANDIDA FAMATA', 'CANDIDA GLABRATA',
    'CANDIDA GUILLIERMONDII', 'CANDIDA KEFYR', 'CANDIDA KRUSEI', 'ZZZCANDIDA PARAPSILOSIS',
    'CANDIDA LIPOL', 'CANDIDA PARAPSILOSIS COMPLEX', 'CANDIDA TROPICALIS'
]


In [ ]:
# Filter and clean
# Bug fix: merge on ['hospital', 'MRN'] instead of just ['MRN'].
# MRN is hospital-specific — BWH MRN 12345 ≠ MGH MRN 12345. Merging on MRN
# alone would cross-match patients across hospitals and attach wrong prior organisms.
previous_infecting_organisms = micro.merge(mrn_map,
                                           on = ['hospital', 'MRN'])

previous_infecting_organisms['previous_organisms'] = (
    previous_infecting_organisms['org_clean'].str.strip().replace('', None)
)
previous_infecting_organisms['prior_infecting_organism_recorded_time'] = previous_infecting_organisms['result_datetime']

# Define organism categories
def categorize_organism(organism):
    org = organism.upper() if pd.notnull(organism) else ''
    if 'ESCHERICHIA' in org: return 'Escherichia'
    if 'KLEBSIELLA' in org: return 'Klebsiella'
    if 'ENTEROCOCCUS' in org: return 'Enterococcus'
    if 'COAG NEGATIVE' in org: return 'CONS'
    if 'STAPH' in org and 'COAG NEGATIVE' not in org: return 'Staphylococcus'
    if 'PROTEUS' in org: return 'Proteus'
    if 'ENTEROBACTER' in org: return 'Enterobacter'
    if 'CITROBACTER' in org: return 'Citrobacter'
    if 'STREPTOCOCC' in org: return 'Streptococcus'
    if 'PSEUDOMONAS' in org: return 'Pseudomonas'
    if 'ACINETOBACTER' in org: return 'Acinetobacter'
    if 'PROVIDENCIA' in org: return 'Providencia'
    if 'SERRATIA' in org: return 'Serratia'
    if 'MORGANELLA' in org: return 'Morganella'
    if 'STENOTROPHOMONAS' in org: return 'Stenotrophomonas'
    if 'CANDIDA' in org: return 'Candida'
    return None

previous_infecting_organisms['prior_organism'] = previous_infecting_organisms['previous_organisms'].apply(categorize_organism)

# Select necessary columns
previous_infecting_organisms_category = previous_infecting_organisms[
    ['PatientID', 'prior_infecting_organism_recorded_time', 'prior_organism']
]

In [ ]:
# Merge with cohort

previous_infecting_organisms_category = previous_infecting_organisms_category.drop_duplicates()

merged_df = base_cohort.merge(
    previous_infecting_organisms_category,
    left_on='anon_id',
    right_on='PatientID',
    how='left'
)

# Filter for positive time difference
merged_df['prior_infecting_organism_days_to_culture'] = (
    (pd.to_datetime(merged_df['order_time_jittered_utc']) - pd.to_datetime(merged_df['prior_infecting_organism_recorded_time'])).dt.days
)

merged_df = merged_df[merged_df['prior_infecting_organism_days_to_culture'] > 0]


In [ ]:
# Final selection and ordering
final_df = merged_df[[
    'anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc',
    'prior_infecting_organism_days_to_culture', 'prior_infecting_organism_recorded_time', 'prior_organism'
]].drop_duplicates()

final_df = final_df.sort_values(by=[
    'anon_id', 'pat_enc_csn_id_coded', 'order_proc_id_coded', 'order_time_jittered_utc', 
    'prior_organism', 'prior_infecting_organism_recorded_time'
])

final_df

In [ ]:
final_df.to_csv(f'{WORK_DIR}/microbiology_cultures_prior_infecting_organism_augmented.csv',
                index = False)

## Deidentification

In [ ]:
# Load the saved mappings (assume you saved as CSVs before)
anon_map_df = pd.read_csv(f'{DEID_DIR}/anon_id_map.csv')
enc_map_df = pd.read_csv(f'{DEID_DIR}/enc_id_map.csv')
order_proc_map_df = pd.read_csv(f'{DEID_DIR}/order_proc_id_map.csv')

In [ ]:
# Convert to Dict
anon_id_map = dict(zip(anon_map_df['anon_id'], anon_map_df['anon_id_deid']))
enc_id_map = dict(zip(enc_map_df['pat_enc_csn_id_coded'], enc_map_df['pat_enc_csn_id_deid']))
order_proc_map = dict(zip(order_proc_map_df['order_proc_id_coded'], order_proc_map_df['order_proc_id_deid']))

# Apply the mappings
final_table_deid = final_df.copy()
final_table_deid['anon_id'] = final_table_deid['anon_id'].map(anon_id_map)
final_table_deid['pat_enc_csn_id_coded'] = final_table_deid['pat_enc_csn_id_coded'].map(enc_id_map)
final_table_deid['order_proc_id_coded'] = final_table_deid['order_proc_id_coded'].map(order_proc_map)

final_table_deid

In [ ]:
final_table_deid.to_csv(f'{DEID_DIR}/microbiology_cultures_prior_infecting_organism_augmented.csv',
                        index = False)